# AI Agents - UC2 DQN
### R. A.
## Lab: Naive DQN on CartPole-v1

This notebook trains a minimal **Deep Q-Network (DQN)** agent to balance the pole in `CartPole-v1`.

It is intentionally a **naive** implementation — it is missing the two techniques that make DQN stable:

- **No replay buffer**: the network is updated on every single transition, in the exact order it was experienced. Consecutive transitions are highly correlated, which makes gradient updates noisy and unstable.
- **No target network**: the TD target is bootstrapped from the *same* network that is being updated. Since the target moves every step the weights change, training can oscillate or diverge.

Seeing this version struggle (or learn unevenly) first makes it much clearer *why* replay buffers and target networks were introduced in the original DQN paper — that will be the next notebook in this series.

## Setup

The `KMP_DUPLICATE_LIB_OK` / `OMP_NUM_THREADS` / `MKL_NUM_THREADS` environment variables must be set **before** `numpy`/`torch` are imported. On macOS, conda's `numpy`/`llvm-openmp` and PyTorch's bundled `libiomp5` can both try to initialize OpenMP, which crashes the process with a duplicate runtime error. `torch.set_num_threads(1)` keeps this tiny MLP from oversubscribing CPU threads.

In [2]:
import random
import os
import time

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

torch.set_num_threads(1)

## Hyperparameters

- `GAMMA`: discount factor for future rewards.
- `LR`: Adam learning rate.
- `EPSILON_START` / `EPSILON_END` / `EPSILON_DECAY_STEPS`: linear epsilon-greedy schedule — epsilon starts at `1.0` (fully random) and decays linearly to `0.01` over the first `EPSILON_DECAY_STEPS` environment steps, then stays at the floor.
- `NUM_STEPS`: total number of **environment steps** (not episodes) to train for.
- `HIDDEN_SIZE`: width of the Q-network's hidden layers.

In [3]:
GAMMA = 0.99
LR = 1e-3
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY_STEPS = 20000
NUM_STEPS = 50000
HIDDEN_SIZE = 128

## Q-network

A small MLP that maps a state (4 numbers for CartPole) to one Q-value per discrete action (2 for CartPole: push left / push right). Two hidden layers of size `HIDDEN_SIZE` with ReLU activations.

In [4]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, HIDDEN_SIZE),
            nn.ReLU(),
            nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
            nn.ReLU(),
            nn.Linear(HIDDEN_SIZE, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## Epsilon-greedy action selection

With probability `epsilon`, take a random action (**explore**). Otherwise, pick the action with the highest predicted Q-value from the network (**exploit**). `torch.no_grad()` is used because action selection should not affect gradients.

In [5]:
def select_action(q_net: QNetwork, state: np.ndarray, epsilon: float, n_actions: int) -> int:
    if random.random() < epsilon:
        return random.randrange(n_actions)
    with torch.no_grad():
        state_t = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = q_net(state_t)
        return int(torch.argmax(q_values, dim=1).item())

## Training loop

For every single environment step:

1. Pick an action with the current epsilon-greedy policy (epsilon decays linearly over time).
2. Step the environment and observe `(reward, next_state, done)`.
3. Compute the **TD target**:
   - If the episode ended: `target = reward`.
   - Otherwise: `target = reward + GAMMA * max_a' Q(next_state, a')`, bootstrapped from the **same** online network — this is the "naive" part, since a proper DQN uses a separate, more slowly-updated target network here.
4. Compute the predicted `Q(state, action)` and take an MSE loss against the target.
5. Backpropagate and update the network with Adam — **every single step**, with no replay buffer batching.

Episode rewards are collected and a rolling average of the last 20 episodes is printed every 10 episodes, so you can watch (unstable) learning progress.

In [6]:
def train():
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    q_net = QNetwork(obs_dim, n_actions)
    optimizer = optim.Adam(q_net.parameters(), lr=LR)

    episode_rewards = []
    episode_reward = 0.0
    state, _ = env.reset()

    i = 0
    while i < NUM_STEPS:
        epsilon = max(
            EPSILON_END,
            EPSILON_START - (EPSILON_START - EPSILON_END) * i / EPSILON_DECAY_STEPS,
        )

        action = select_action(q_net, state, epsilon, n_actions)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        episode_reward += reward
        i += 1

        state_t = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        next_state_t = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

        # TD target: bootstrapped from the same online network (no target net) -> "naive" DQN
        with torch.no_grad():
            if done:
                target = torch.tensor([reward], dtype=torch.float32)
            else:
                next_q_max = torch.max(q_net(next_state_t), dim=1)[0]
                target = reward + GAMMA * next_q_max

        q_values = q_net(state_t)
        q_action = q_values.gather(1, torch.tensor([[action]])).squeeze(1)

        loss = (target - q_action).pow(2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        state = next_state

        if done:
            episode_rewards.append(episode_reward)
            episode_reward = 0.0
            state, _ = env.reset()

            avg_reward = np.mean(episode_rewards[-20:])
            if len(episode_rewards) % 10 == 0:
                print(
                    f"Step {i}/{NUM_STEPS} episode={len(episode_rewards)} "
                    f"reward={episode_rewards[-1]:.1f} avg20={avg_reward:.1f} epsilon={epsilon:.3f}"
                )

    env.close()
    return q_net

## Run training

Trains for `NUM_STEPS = 50000` environment steps (roughly 15-20 seconds on a modern CPU). Expect a noisy, uneven reward curve — that instability is the point of this "naive" version.

In [7]:
trained_q_net = train()

Step 257/50000 episode=10 reward=23.0 avg20=25.7 epsilon=0.987
Step 449/50000 episode=20 reward=25.0 avg20=22.4 epsilon=0.978
Step 661/50000 episode=30 reward=17.0 avg20=20.2 epsilon=0.967
Step 856/50000 episode=40 reward=28.0 avg20=20.4 epsilon=0.958
Step 1089/50000 episode=50 reward=31.0 avg20=21.4 epsilon=0.946
Step 1299/50000 episode=60 reward=25.0 avg20=22.1 epsilon=0.936
Step 1557/50000 episode=70 reward=22.0 avg20=23.4 epsilon=0.923
Step 1754/50000 episode=80 reward=23.0 avg20=22.8 epsilon=0.913
Step 1923/50000 episode=90 reward=15.0 avg20=18.3 epsilon=0.905
Step 2205/50000 episode=100 reward=15.0 avg20=22.6 epsilon=0.891
Step 2452/50000 episode=110 reward=34.0 avg20=26.4 epsilon=0.879
Step 2624/50000 episode=120 reward=34.0 avg20=20.9 epsilon=0.870
Step 2854/50000 episode=130 reward=26.0 avg20=20.1 epsilon=0.859
Step 3128/50000 episode=140 reward=65.0 avg20=25.2 epsilon=0.845
Step 3492/50000 episode=150 reward=13.0 avg20=31.9 epsilon=0.827
Step 3724/50000 episode=160 reward=14.

## Inspect the network's input and output

Before watching the agent play, it helps to see exactly what goes **in** and **out** of the Q-network at each decision:

- **Input**: the raw state `(x, x_dot, theta, theta_dot)` from the environment, converted to a `(1, 4)` float32 tensor (batch size 1).
- **Output**: a `(1, 2)` tensor of Q-values, one estimated expected-return per action (`Q(left)`, `Q(right)`).
- The greedy action is simply `argmax` over the output Q-values.

The cell below steps through a short greedy rollout and prints the state fed in and the Q-values returned at every step, so you can watch how the network's output reacts as the state changes (e.g. Q-values should favor pushing toward the side that counteracts the pole's tilt).

In [8]:
def inspect_q_values(q_net: QNetwork, state: np.ndarray):
    """Returns the exact input tensor fed to q_net and the output Q-values tensor it produces."""
    state_t = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        q_values = q_net(state_t)
    return state_t, q_values


inspect_env = gym.make("CartPole-v1")
state, _ = inspect_env.reset(seed=0)

header = f"{'step':>4} | {'input state (x, x_dot, theta, theta_dot)':42} | {'input shape':12} | {'Q(left)':>8} {'Q(right)':>9} | action"
print(header)
print("-" * len(header))

for step in range(10):
    state_t, q_values = inspect_q_values(trained_q_net, state)
    action = int(torch.argmax(q_values, dim=1).item())
    q_left, q_right = q_values.squeeze(0).tolist()
    state_str = np.array2string(state, precision=3, suppress_small=True)
    print(f"{step:4d} | {state_str:42} | {str(tuple(state_t.shape)):12} | {q_left:8.3f} {q_right:9.3f} | {action} ({'left' if action == 0 else 'right'})")

    state, reward, terminated, truncated, _ = inspect_env.step(action)
    if terminated or truncated:
        print(f"(episode ended after step {step})")
        break

inspect_env.close()

step | input state (x, x_dot, theta, theta_dot)   | input shape  |  Q(left)  Q(right) | action
----------------------------------------------------------------------------------------------
   0 | [ 0.014 -0.023 -0.046 -0.048]              | (1, 4)       |   72.850    77.217 | 1 (right)
   1 | [ 0.013  0.173 -0.047 -0.355]              | (1, 4)       |   73.853    70.133 | 0 (left)
   2 | [ 0.017 -0.022 -0.054 -0.078]              | (1, 4)       |   73.575    76.294 | 1 (right)
   3 | [ 0.016  0.174 -0.056 -0.387]              | (1, 4)       |   74.380    68.872 | 0 (left)
   4 | [ 0.02  -0.02  -0.063 -0.112]              | (1, 4)       |   74.287    75.039 | 1 (right)
   5 | [ 0.019  0.176 -0.066 -0.424]              | (1, 4)       |   74.860    67.258 | 0 (left)
   6 | [ 0.023 -0.018 -0.074 -0.153]              | (1, 4)       |   74.937    73.425 | 0 (left)
   7 | [ 0.022 -0.212 -0.077  0.116]              | (1, 4)       |   73.184    78.075 | 1 (right)
   8 | [ 0.018 -0.016 -0.075 -

## Watch the trained agent

Runs the learned **greedy** policy (`epsilon=0`, no exploration) for a few episodes with `render_mode="human"`, which opens a live pygame window so you can visually inspect behavior.

> **Note:** `render_mode="human"` requires a local display. It works when running this notebook on your own machine, but it will fail (or need a virtual display such as `Xvfb`) on a headless remote server or SSH session without a display.

In [9]:
def run_trained_agent(q_net: QNetwork, episodes: int = 5):
    env = gym.make("CartPole-v1", render_mode="human")
    n_actions = env.action_space.n

    for ep in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            action = select_action(q_net, state, epsilon=0.0, n_actions=n_actions)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        print(f"[Render] Episode {ep + 1} reward={total_reward:.1f}")

    env.close()

## Run it

Equivalent to the original script's `if __name__ == "__main__":` block: train, then watch the trained agent play.

In [ ]:
run_trained_agent(trained_q_net)

[Render] Episode 1 reward=172.0
[Render] Episode 2 reward=178.0
[Render] Episode 3 reward=181.0
[Render] Episode 4 reward=176.0
[Render] Episode 5 reward=164.0


: 